# Early Moment Search (Label '1')

This notebook replicates a subset of the pipeline to:
- Load model/SAEs and generate the prompt continuation for a given `prompt_idx` and `inter_token_id`.
- Load clusters from a prior run (`outputs/prompt_15/token_294/clusters.json`).
- Run the per-position earliest planning search restricted to label "1" (see logic in `plan_trace/pipeline.py:922-973`).


In [ ]:
# Imports and configuration
import re
import sys 
import json
import time
import torch
from pathlib import Path
sys.path.append("../")
from plan_trace.utils import load_model, load_pretrained_saes, cleanup_cuda
from plan_trace.steering import run_steering_sweep
from plan_trace.ood_detect import label_steering_clusters

# Config (adjust as needed)
model_name = "gemma-2-2b-it"
device = "cuda"
use_custom_cache = True  # toggle as needed; mirrors --use-custom-cache


data_path = "../data/first_100_passing_examples.json"
prompt_idx = 15
inter_token_id = 297  # token_pred_idx to anchor prefix for analysis
stop_token_id = 1917  # token for ```
coeff_grid = list(range(-100, 0, 20))

def build_prompt(entry: dict) -> str:
    return (
        "You are an expert Python programmer, and here is your task: "
        f"{entry['prompt']} Your code should pass these tests:\n\n"
        + "\n".join(entry["test_list"]) + "\nWrite your code below starting with \"```python\" and ending with \"```\".\n```python\n"
    )

In [ ]:
# Load model/SAEs and generate tokens
with open(data_path, 'r') as f:
    data = json.load(f)

entry = data[prompt_idx]
prompt = build_prompt(entry)

print("Loading model and SAEs...")
model = load_model(model_name, device=device, use_custom_cache=use_custom_cache, dtype=torch.bfloat16)
layers = list(range(model.cfg.n_layers))
saes = load_pretrained_saes(
    layers=layers,
    release="gemma-scope-2b-pt-mlp-canonical",
    width="16k",
    device=device,
    canon=True,
)


Loading model and SAEs...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded pretrained model gemma-2-2b-it into HookedTransformer
Generating full sequence...
Prompt length: 217, Total length: 308
Baseline continuation preview:  sorted(marks, key=lambda x: x[1])
...


In [7]:
inter_token_id = 297
print("Generating full sequence...")
toks_BL = model.to_tokens(prompt).to(device)
out_BL = toks_BL.clone()

while out_BL.shape[-1] - toks_BL.shape[-1] < 150:
    with torch.no_grad():
        logits_V = model(out_BL)[0, -1]
    next_id = logits_V.argmax(-1).item()
    del logits_V
    cleanup_cuda()
    if next_id == stop_token_id:
        break
    out_BL = torch.cat([out_BL, torch.tensor([[next_id]], device=device)], dim=1)

inter_toks_BL = out_BL[:, :inter_token_id]
baseline_suffix = model.to_string(out_BL[0, inter_token_id:])
print(f"Prompt length: {toks_BL.shape[-1]}, Total length: {out_BL.shape[-1]}")
print(f"Baseline continuation preview: {baseline_suffix[:100]}...")

Generating full sequence...
Prompt length: 217, Total length: 308
Baseline continuation preview: , key=lambda x: x[1])
...


In [8]:
# Load clusters and restrict to label '1'
clusters_path = Path("../outputs/per_pos/test/prompt_15/token_297/clusters.json")
with open(clusters_path, 'r') as f:
    saved_pair_dict = json.load(f)

print("Available labels:", list(saved_pair_dict.keys()))
if '1' not in saved_pair_dict:
    raise ValueError("Label '1' not found in clusters.json")

saved_pair_dict = {'1': saved_pair_dict['1']}
print("Using labels:", list(saved_pair_dict.keys()))

# Collect sorted unique token positions for label '1'
positions = sorted({
    tok_pos
    for (_, _, tok_positions) in saved_pair_dict['1']
    for tok_pos in tok_positions
})
print(f"Candidate token positions for label '1': {positions[:20]}{'...' if len(positions) > 20 else ''}")


Available labels: ['1', 'x', 'key']
Using labels: ['1']
Candidate token positions for label '1': [2, 3, 4, 22, 26, 27, 28, 30, 31, 34, 36, 40, 43, 44, 46, 48, 50, 51, 58, 65]...


In [15]:
len(positions)

70

In [16]:
# Per-position sweep from the end, print steered generations (strings) − fixed printing
n_positions = 100
coeff_grid_local = list(range(-400, 0, 100))
max_tokens_local = 50

assert list(saved_pair_dict.keys()) == ['1'], "saved_pair_dict must be filtered to label '1'"

positions_desc = sorted({
    tok_pos
    for (_, _, tok_positions) in saved_pair_dict['1']
    for tok_pos in tok_positions
}, reverse=False)

earliest_position = None
earliest_position_steering_results = None

for tok_pos in positions_desc[:n_positions]:
    # Build filtered dict limited to this exact position
    sub = []
    for (li, latent_i, tok_positions) in saved_pair_dict['1']:
        if tok_pos in tok_positions:
            sub.append((li, latent_i, [tok_pos]))
    if not sub:
        continue
    filtered = {'1': sub}

    # Show which (layer, latent) pairs will be steered
    layer_latents = [(li, latent_i) for (li, latent_i, _) in sub]
    preview_pairs = layer_latents if len(layer_latents) <= 20 else layer_latents[:20] + [("...", "...")]
    print(f"\n=== Position {tok_pos} ===")
    print(f"Steering {len(layer_latents)} (layer, latent) pairs -> {preview_pairs}")

    # Run sweep with return_tokens=False to get strings
    pos_steering = run_steering_sweep(
        model=model,
        saes=saes,
        inter_toks_BL=inter_toks_BL,
        saved_pair_dict=filtered,
        baseline_text=baseline_suffix,
        coeff_grid=coeff_grid_local,
        stop_tok=stop_token_id,
        max_tokens=max_tokens_local,
        return_tokens=False,
    )

    # Label results
    pos_labels = label_steering_clusters(pos_steering, model=model, prefix_tokens_2d=inter_toks_BL)
    label_map = {k: v['final_label'] for k, v in pos_labels.items()}
    print(f"Labels: {label_map}")

    # Print baseline and steered generations (precompute previews to avoid backslashes in f-strings)
    base_text = pos_steering.get('1', {}).get('base_text', '')
    base_preview = (base_text[:200] if isinstance(base_text, str) else str(base_text)) \
        .replace("\n", " ")
    print("Baseline (preview): " + base_preview)

    for item in pos_steering.get('1', {}).get('steered', []):
        coeff = item.get('coeff')
        steered_text = item.get('steered_text', '')
        if hasattr(steered_text, "tolist"):
            try:
                steered_text = model.to_string(steered_text.tolist())
            except Exception:
                steered_text = str(steered_text)
        steered_preview = (steered_text[:200] if isinstance(steered_text, str) else str(steered_text)) \
            .replace("\n", " ")
        print(f"  coeff {coeff:>4}: " + steered_preview)

    if any(v == "Plan" for v in label_map.values()):
        earliest_position = tok_pos
        earliest_position_steering_results = pos_steering
        print(f"--> Found earliest Plan at position {tok_pos}")
        break

print(f"\nEarliest planning position (label '1'): {earliest_position}")


=== Position 2 ===
Steering 1 (layer, latent) pairs -> [(6, 10832)]
Labels: {'1': 'Not planning'}
Baseline (preview): , key=lambda x: x[1]) 
  coeff -400: 
  coeff -300: 
  coeff -200: 
  coeff -100: 

=== Position 3 ===
Steering 1 (layer, latent) pairs -> [(4, 4189)]
Labels: {'1': 'Not planning'}
Baseline (preview): , key=lambda x: x[1]) 
  coeff -400: 
  coeff -300: 
  coeff -200: 
  coeff -100: 

=== Position 4 ===
Steering 1 (layer, latent) pairs -> [(4, 11586)]
Labels: {'1': 'Not planning'}
Baseline (preview): , key=lambda x: x[1]) 
  coeff -400: 
  coeff -300: 
  coeff -200: 
  coeff -100: 

=== Position 22 ===
Steering 3 (layer, latent) pairs -> [(7, 2262), (7, 7643), (3, 5860)]
Labels: {'1': 'Not planning'}
Baseline (preview): , key=lambda x: x[1]) 
  coeff -400: 
  coeff -300: 
  coeff -200: 
  coeff -100: 

=== Position 26 ===
Steering 7 (layer, latent) pairs -> [(4, 5502), (4, 5876), (6, 16150), (7, 7643), (3, 5226), (3, 8364), (5, 2524)]
Labels: {'1': 'Not planning'}
Basel

# earliest position for this task

```
=== Position 296 ===
Steering 2 (layer, latent) pairs -> [(21, 2984), (11, 8764)]
Labels: {'1': 'Plan'}
Baseline (preview): , key=lambda x: x[1]) 
  coeff -400: [::], key=lambda x: x[1]) 
  coeff -300: ) 
  coeff -200: 
  coeff -100: 
--> Found earliest Plan at position 296

Earliest planning position (label '1'): 296
```

None of the previous tokens did it.

# activation based steering

In [17]:
tok_pos = 296
sub = []
for (li, latent_i, tok_positions) in saved_pair_dict['1']:
    if tok_pos in tok_positions:
        sub.append((li, latent_i, [tok_pos]))
filtered = {'1': sub}
print(filtered)

{'1': [(21, 2984, [296]), (11, 8764, [296])]}


In [25]:
# Get SAE caches (feature activations per layer) for the current input
from plan_trace.hooks import run_with_saes
from plan_trace.circuit_discovery import get_saes_cache
import torch

# Example targets (label '1'): [(layer_idx, latent_idx, [token_positions]), ...]
targets = {'1': [(21, 2984, [296]), (11, 8764, [296])]}

# 1) Run a forward pass with SAEs caching activations
model.reset_hooks(including_permanent=True)
with torch.no_grad():
    _, saes = run_with_saes(
        model,
        saes,
        inter_toks_BL,
        calc_error=True,
        use_error=True,
        cache_sae_activations=True,   # <-- fills sae.feature_acts
    )

# 2) Extract caches (feature_acts is [L, S] for each SAE)
clean_sae_cache, clean_error_cache, _, _ = get_saes_cache(saes)
cleanup_cuda()

clean_sae_cache['blocks.0.hook_mlp_out'].shape

torch.Size([1, 297, 16384])

In [28]:
print(model.to_string(inter_toks_BL)[0])

<bos>You are an expert Python programmer, and here is your task: Write a function to sort a list of tuples using the second value of each tuple. Your code should pass these tests:

assert subject_marks([('English', 88), ('Science', 90), ('Maths', 97), ('Social sciences', 82)])==[('Social sciences', 82), ('English', 88), ('Science', 90), ('Maths', 97)]
assert subject_marks([('Telugu',49),('Hindhi',54),('Social',33)])==([('Social',33),('Telugu',49),('Hindhi',54)])
assert subject_marks([('Physics',96),('Chemistry',97),('Biology',45)])==([('Biology',45),('Physics',96),('Chemistry',97)])
Write your code below starting with "```python" and ending with "```".
```python
def subject_marks(marks):
    """
    Sorts a list of tuples by the second value of each tuple.

    Args:
        marks: A list of tuples, where each tuple represents a subject and its corresponding mark.

    Returns:
        A new list of tuples, sorted by the second value of each tuple.
    """
    return sorted(marks


In [21]:
# 3) Read activations for your targets at inter_token_id (and/or explicit token positions)
def read_latent_act(layer_idx: int, latent_idx: int, token_idx: int) -> float:
    hook_name = saes[layer_idx].cfg.hook_name
    acts_LS = clean_sae_cache[hook_name]  # shape [L, S]
    # ensure on CPU for .item() if needed
    return acts_LS[0, token_idx, latent_idx].item()
    
# Example: use the provided token positions
acts_by_target = {}
for (layer_idx, latent_idx, token_positions) in targets['1']:
    for pos in token_positions:
        key = (layer_idx, latent_idx, pos)
        acts_by_target[key] = read_latent_act(layer_idx, latent_idx, pos)

# Or: use a specific inter_token_id instead of token_positions
# inter_token_id = 296  # set this to your analysis position
# for (layer_idx, latent_idx, _) in targets['1']:
#     key = (layer_idx, latent_idx, inter_token_id)
#     acts_by_target[key] = read_latent_act(layer_idx, latent_idx, inter_token_id)

print("Latent activations:", acts_by_target)

Latent activations: {(21, 2984, 296): 3.2064757347106934, (11, 8764, 296): 2.26546049118042}


In [50]:
# Flip selected SAE latent activations by -1 at specific token positions,
# while also re-adding the cached error via use_mean_error=True.

from plan_trace.hooks import build_sae_hook_fn
from plan_trace.hooks import run_with_saes
from plan_trace.circuit_discovery import get_saes_cache
from plan_trace.steering import generate_once
import torch

# Targets: (layer_idx, latent_idx, token_idx)
targets = [
    (21, 2984, 296),
    (11, 8764, 296),
]
coeff_act = -168

# 1) Cache pass to get feature_acts and error_term
model.reset_hooks(including_permanent=True)
with torch.no_grad():
    _, saes = run_with_saes(
        model,
        saes,
        inter_toks_BL,
        calc_error=True,
        use_error=True,
        cache_sae_activations=True,
    )

# Map mean_error to the cached error so we can add it back later
for sae in saes:
    sae.mean_error = sae.error_term.detach()

# 2) Grab clean activations per layer
clean_sae_cache, clean_error_cache, _, _ = get_saes_cache(saes)

# Ensure acts are [B, L, S]
def ensure_bls(t):
    if t.dim() == 2:  # [L, S] -> [1, L, S]
        return t.unsqueeze(0)
    return t

# Build per-layer modified activations by flipping sign at the targets
layer_to_acts = {}
for (layer_idx, latent_idx, tok_idx) in targets:
    hook_name = saes[layer_idx].cfg.hook_name
    if layer_idx not in layer_to_acts:
        layer_to_acts[layer_idx] = ensure_bls(clean_sae_cache[hook_name].clone())
    acts = layer_to_acts[layer_idx]
    if 0 <= tok_idx < acts.size(1) and 0 <= latent_idx < acts.size(2):
        acts[0, tok_idx, latent_idx] = -coeff_act * acts[0, tok_idx, latent_idx]

# 3) Build per-layer hooks with fake_activations and use_mean_error=True
bos_id = model.tokenizer.bos_token_id
hooks = []
for layer_idx, sae in enumerate(saes):
    acts_override = layer_to_acts.get(layer_idx, None)
    hook_fn = build_sae_hook_fn(
        sae,
        sequence=inter_toks_BL[0],            # [L]
        bos_token_id=bos_id,
        circuit_mask=None,
        mean_mask=False,
        cache_masked_activations=False,
        cache_sae_activations=False,
        mean_ablate=False,
        fake_activations=(sae.cfg.hook_layer, acts_override) if acts_override is not None else False,
        calc_error=False,
        use_error=False,
        use_mean_error=True,                  # add back mean_error we set above
    )
    hooks.append((sae.cfg.hook_name, hook_fn))

# 4) Predict only the very next token (no growth)

with torch.no_grad():
    logits = model.run_with_hooks(
        inter_toks_BL, return_type="logits", fwd_hooks=hooks
    )  # [B, L, V]
    next_logits = logits[:, -1, :]           # [B, V]
    next_id = next_logits.argmax(-1).item()  # int
    next_tok = model.to_string(next_id)
model.reset_hooks(including_permanent=True)
print("Next token id:", next_id)
print("Next token str:", next_tok)

# 4) Generate with the interventions (no permanent hooks)
# intervened_text = generate_once(
#     model,
#     inter_toks_BL=inter_toks_BL,
#     stop_tok=stop_token_id,
#     hooks=hooks,
#     max_tokens=50,
#     device=str(next(model.parameters()).device),
#     return_tokens=False,
# )
# print("Intervened continuation:", intervened_text)

Next token id: 77056
Next token str: )**


needed coefficient of -200, which is crazy

# act steered multi generation is not working, need to fix

In [ ]:
# Intervene in SAE latent activations by forcing specific values at specific token positions
# Requirements in scope:
# - model, saes, inter_toks_BL (prefix tokens), stop_token_id
# - You can obtain saes via run_with_saes(..., cache_sae_activations=True) if needed

import torch

# 1) Define desired overrides: (layer_idx, latent_idx, token_idx) -> target_value
# Example flipping signs of measured activations at token 296:
desired_overrides = {
    (21, 2984, 296): -3.2064757347106934,
    (11, 8764, 296): -2.26546049118042,
}

# Group by layer for efficient hooking
overrides_by_layer = {}
for (layer_idx, latent_idx, tok_idx), target_value in desired_overrides.items():
    overrides_by_layer.setdefault(layer_idx, []).append((latent_idx, tok_idx, float(target_value)))

# 2) Build hooks per layer that:
#    - encode -> feature_acts [B, L, S]
#    - override feature_acts[:, tok_idx, latent_idx] = target_value
#    - decode and merge with original activations (respecting BOS mask)
bos_id = model.tokenizer.bos_token_id
tokens = inter_toks_BL  # [B, L], B=1
assert tokens.dim() == 2 and tokens.size(0) == 1, "Expected batch size 1"

seq_mask = torch.ones_like(tokens, dtype=torch.bool)
seq_mask[tokens == bos_id] = False  # keep original at BOS positions
seq_mask_expanded = seq_mask.unsqueeze(-1)  # [B, L, 1]

def make_intervention_hook(layer_idx: int):
    sae = saes[layer_idx]
    layer_overrides = overrides_by_layer[layer_idx]  # List[(latent_idx, tok_idx, target_value)]

    def hook_fn(value: torch.Tensor, hook):
        # value: [B, L, D_model]
        feature_acts = sae.encode(value)  # [B, L, S]
        # Apply overrides
        for latent_idx, tok_idx, target_value in layer_overrides:
            # Safety: bounds check
            if 0 <= tok_idx < feature_acts.size(1) and 0 <= latent_idx < feature_acts.size(2):
                feature_acts[:, tok_idx, latent_idx] = torch.tensor(target_value, device=feature_acts.device, dtype=feature_acts.dtype)

        # Decode and merge (mirror standard SAE hook behavior)
        out = sae.decode(feature_acts)  # [B, L, D_model]
        updated_value = torch.where(seq_mask_expanded.expand_as(value), out, value)
        return updated_value

    return hook_fn

# 3) Register hooks for all layers you want to intervene on
for layer_idx in overrides_by_layer.keys():
    hook_name = saes[layer_idx].cfg.hook_name
    model.add_hook(hook_name, make_intervention_hook(layer_idx))

# 4) Generate with interventions
from plan_trace.steering import generate_once

intervened_text = generate_once(
    model,
    inter_toks_BL=inter_toks_BL,
    stop_tok=stop_token_id,
    hooks=None,             # we already added permanent hooks above
    max_tokens=50,
    device=str(next(model.parameters()).device),
    return_tokens=False,
)
print("Intervened continuation:", intervened_text)

# 5) Cleanup hooks after use
model.reset_hooks(including_permanent=True)